## 3.2 Sparse Retrieval (BM25)

BM25 é uma versão melhorada do TF-IDF — o algoritmo clássico de busca textual que o Google usava antes das redes neurais.

**Como funciona:** pondera palavras por:
- **TF (Term Frequency)**: quantas vezes a palavra aparece no documento
- **IDF (Inverse Document Frequency)**: palavras raras recebem mais peso que palavras comuns
- **Normalização por tamanho**: documentos longos não ganham vantagem injusta

**Ponto forte:** encontra documentos que contêm exatamente as palavras da query. Para queries como "CUDA out of memory error", BM25 vai direto nos documentos que contêm essas palavras específicas.

**Ponto fraco:** completamente cego a semântica. "cachorro" e "cão" são palavras totalmente diferentes para o BM25. Um documento sobre "caninos domésticos" não aparece para uma query sobre "cachorros".

**Na prática:** BM25 é imbatível para buscas keyword-based (código, nomenclaturas técnicas, nomes próprios). Dense é melhor para linguagem natural e semântica.

In [ ]:
## 3.2 Sparse Retrieval (BM25)

BM25 é uma versão melhorada do TF-IDF — o algoritmo clássico de busca textual que o Google usava antes das redes neurais.

**Como funciona:** pondera palavras por:
- **TF (Term Frequency)**: quantas vezes a palavra aparece no documento
- **IDF (Inverse Document Frequency)**: palavras raras recebem mais peso que palavras comuns
- **Normalização por tamanho**: documentos longos não ganham vantagem injusta

**Ponto forte:** encontra documentos que contêm exatamente as palavras da query. Para queries como "CUDA out of memory error", BM25 vai direto nos documentos que contêm essas palavras específicas.

**Ponto fraco:** completamente cego a semântica. "cachorro" e "cão" são palavras totalmente diferentes para o BM25. Um documento sobre "caninos domésticos" não aparece para uma query sobre "cachorros".

**Na prática:** BM25 é imbatível para buscas keyword-based (código, nomenclaturas técnicas, nomes próprios). Dense é melhor para linguagem natural e semântica.

## 3.1 Dense Retrieval (Embedding-based)

O retrieval denso é o que você já conhece: embedding da query → similaridade cosine → top-K.

**Ponto forte:** entende *semântica*. "cachorro" e "cão" são similares. "como resolver problema X" encontra documentos que falam de "soluções para X" mesmo sem usar exatamente essas palavras.

**Ponto fraco:** para queries muito específicas com termos técnicos exatos (números de erro, códigos de produto, nomes próprios incomuns), o embedding pode não capturar bem a especificidade. O modelo foi treinado em linguagem geral — termos muito de nicho podem ter representações imprecisas.

**Também falha com negação:** "o que NÃO é machine learning?" — o embedding de "NÃO machine learning" fica próximo de embeddings de machine learning, porque o modelo entende o conceito, não a negação.

In [ ]:
def dense_retrieve(query, top_k=5):
    q_vec = model.encode(query, normalize_embeddings=True)
    results = client.query_points('retrieval_demo', query=q_vec.tolist(), limit=top_k, with_payload=True).points
    return [(r.id, r.score, r.payload['texto']) for r in results]

# Casos onde dense se sai bem: busca semantica
queries_semanticas = [
    ('redes neurais para linguagem', 'Transformers, BERT, GPT — entende contexto'),
    ('encontrar itens similares rapidamente', 'HNSW, cosine similarity — captura intenção'),
]

print('DENSE RETRIEVAL — casos onde se sai bem:')
for query, comentario in queries_semanticas:
    results = dense_retrieve(query, top_k=3)
    print(f'\nQuery: "{query}" ({comentario})')
    for i, (idx, score, texto) in enumerate(results):
        print(f'  [{i+1}] {score:.3f}: {texto[:70]}')

## 3.2 Sparse Retrieval (BM25)

BM25 é uma versão melhorada do TF-IDF — o algoritmo clássico de busca textual que o Google usava antes das redes neurais.

**Como funciona:** pondera palavras por:
- **TF (Term Frequency)**: quantas vezes a palavra aparece no documento
- **IDF (Inverse Document Frequency)**: palavras raras recebem mais peso que palavras comuns
- **Normalização por tamanho**: documentos longos não ganham vantagem injusta

**Ponto forte:** encontra documentos que contêm exatamente as palavras da query. Para queries como "CUDA out of memory error", BM25 vai direto nos documentos que contêm essas palavras específicas.

**Ponto fraco:** completamente cego a semântica. "cachorro" e "cão" são palavras totalmente diferentes para o BM25. Um documento sobre "caninos domésticos" não aparece para uma query sobre "cachorros".

**Na prática:** BM25 é imbatível para buscas keyword-based (código, nomenclaturas técnicas, nomes próprios). Dense é melhor para linguagem natural e semântica.

In [ ]:
## 3.2 Sparse Retrieval (BM25)

BM25 é uma versão melhorada do TF-IDF — o algoritmo clássico de busca textual que o Google usava antes das redes neurais.

**Como funciona:** pondera palavras por:
- **TF (Term Frequency)**: quantas vezes a palavra aparece no documento
- **IDF (Inverse Document Frequency)**: palavras raras recebem mais peso que palavras comuns
- **Normalização por tamanho**: documentos longos não ganham vantagem injusta

**Ponto forte:** encontra documentos que contêm exatamente as palavras da query. Para queries como "CUDA out of memory error", BM25 vai direto nos documentos que contêm essas palavras específicas.

**Ponto fraco:** completamente cego a semântica. "cachorro" e "cão" são palavras totalmente diferentes para o BM25. Um documento sobre "caninos domésticos" não aparece para uma query sobre "cachorros".

**Na prática:** BM25 é imbatível para buscas keyword-based (código, nomenclaturas técnicas, nomes próprios). Dense é melhor para linguagem natural e semântica.

## 3.3 Hybrid Search com RRF

Hybrid search combina os rankings de dense e sparse para capturar o melhor dos dois mundos.

**RRF (Reciprocal Rank Fusion)** é o algoritmo de fusão mais popular:

```
score_rrf(doc) = 1/(k + rank_dense) + 1/(k + rank_sparse)
```

Onde `k=60` é uma constante empírica. Um documento que aparece em 1º no dense e 1º no sparse tem score máximo. Um documento que aparece apenas em um dos rankings ainda contribui com algum score.

**Por que não simplesmente somar os scores?** Scores de sistemas diferentes têm escalas incompatíveis (cosine: 0-1, BM25: 0-∞). RRF usa apenas os *ranks* (posições), não os valores absolutos — isso é escala-invariante.

**O parâmetro `alpha`** controla o peso relativo:
- `alpha=1.0`: só dense
- `alpha=0.0`: só sparse
- `alpha=0.5`: equilíbrio — funciona bem como padrão

In [ ]:
## 3.2 Sparse Retrieval (BM25)

BM25 é uma versão melhorada do TF-IDF — o algoritmo clássico de busca textual que o Google usava antes das redes neurais.

**Como funciona:** pondera palavras por:
- **TF (Term Frequency)**: quantas vezes a palavra aparece no documento
- **IDF (Inverse Document Frequency)**: palavras raras recebem mais peso que palavras comuns
- **Normalização por tamanho**: documentos longos não ganham vantagem injusta

**Ponto forte:** encontra documentos que contêm exatamente as palavras da query. Para queries como "CUDA out of memory error", BM25 vai direto nos documentos que contêm essas palavras específicas.

**Ponto fraco:** completamente cego a semântica. "cachorro" e "cão" são palavras totalmente diferentes para o BM25. Um documento sobre "caninos domésticos" não aparece para uma query sobre "cachorros".

**Na prática:** BM25 é imbatível para buscas keyword-based (código, nomenclaturas técnicas, nomes próprios). Dense é melhor para linguagem natural e semântica.

In [ ]:
## 3.2 Sparse Retrieval (BM25)

BM25 é uma versão melhorada do TF-IDF — o algoritmo clássico de busca textual que o Google usava antes das redes neurais.

**Como funciona:** pondera palavras por:
- **TF (Term Frequency)**: quantas vezes a palavra aparece no documento
- **IDF (Inverse Document Frequency)**: palavras raras recebem mais peso que palavras comuns
- **Normalização por tamanho**: documentos longos não ganham vantagem injusta

**Ponto forte:** encontra documentos que contêm exatamente as palavras da query. Para queries como "CUDA out of memory error", BM25 vai direto nos documentos que contêm essas palavras específicas.

**Ponto fraco:** completamente cego a semântica. "cachorro" e "cão" são palavras totalmente diferentes para o BM25. Um documento sobre "caninos domésticos" não aparece para uma query sobre "cachorros".

**Na prática:** BM25 é imbatível para buscas keyword-based (código, nomenclaturas técnicas, nomes próprios). Dense é melhor para linguagem natural e semântica.

## 3.2 Sparse Retrieval (BM25)

BM25 é uma versão melhorada do TF-IDF — o algoritmo clássico de busca textual que o Google usava antes das redes neurais.

**Como funciona:** pondera palavras por:
- **TF (Term Frequency)**: quantas vezes a palavra aparece no documento
- **IDF (Inverse Document Frequency)**: palavras raras recebem mais peso que palavras comuns
- **Normalização por tamanho**: documentos longos não ganham vantagem injusta

**Ponto forte:** encontra documentos que contêm exatamente as palavras da query. Para queries como "CUDA out of memory error", BM25 vai direto nos documentos que contêm essas palavras específicas.

**Ponto fraco:** completamente cego a semântica. "cachorro" e "cão" são palavras totalmente diferentes para o BM25. Um documento sobre "caninos domésticos" não aparece para uma query sobre "cachorros".

**Na prática:** BM25 é imbatível para buscas keyword-based (código, nomenclaturas técnicas, nomes próprios). Dense é melhor para linguagem natural e semântica.

## Resumo

| Estratégia | Recall semântico | Recall keyword | Diversidade | Complexidade |
|-----------|-----------------|---------------|-------------|-------------|
| Dense | ★★★★★ | ★★☆☆☆ | ★★☆☆☆ | Baixa |
| Sparse (BM25) | ★★☆☆☆ | ★★★★★ | ★★☆☆☆ | Baixa |
| **Hybrid RRF** | **★★★★★** | **★★★★★** | **★★☆☆☆** | **Média** |
| MMR | ★★★★☆ | ★★☆☆☆ | ★★★★★ | Média |

**Hybrid com RRF é o padrão de produção moderno.** Sistemas como Elasticsearch, Qdrant e Weaviate têm suporte nativo a hybrid search exatamente por isso.

**Próximos passos:**
- [04 — Generation Prompts](04_generation_prompts.html): como instruir o LLM para gerar respostas precisas e evitar alucinações